# NB3 — Train basins, derive canonical M2 / M3 / M4 / M5

v7 pipeline. Runs on Kaggle GPU (T4 recommended).

## What this notebook does

1. Trains `N_BASINS` (default 50) independently-seeded MLPs per dataset. Each basin is a full,
   separate training run (own weight initialization, own seed), saved as its own checkpoint.
2. From that single pool of 50 checkpoints, derives four canonical models by varying only how
   many basins and how many MC-Dropout passes are used at inference time:

| Model | Basins used | Passes / basin | Dropout at inference | Total forward evals |
|---|---|---|---|---|
| M2 | basin 1 | 1 | off | 1 |
| M3 | basin 1 | 50 | on | 50 |
| M4 | all 50  | 1 | off | 50 |
| M5 | basin 1-5 | 10 | on | 50 |

M2 and M3 use the **same trained network** — only the inference procedure differs (a single
deterministic pass vs. 50 MC-Dropout passes). This isolates exactly one manipulation (whether
MC-Dropout is switched on at inference) instead of confounding it with a second, independently
trained network, which is what the v6 design did. M3, M4 and M5 all spend the same forward-pass
budget (M x T = 50), which fixes the v6 confound where M4 (Deep Ensemble) used a 5-pass budget
against M3/M5's 100/500-pass budgets — the old M5 > M4 comparison was mostly a comparison of
posterior sample counts, not architectures.

## Uncertainty decomposition (HB-MCD)

For M basins, each run for T MC-Dropout passes, let `p[m,t,n]` be the predicted probability for
sample n, basin m, pass t. Define `p_bar_basin[m,n] = mean_t p[m,t,n]` and
`p_bar[n] = mean_m p_bar_basin[m,n]`. With `H(p) = -p log p - (1-p) log(1-p)`:

```
U_aleatoric[n] = mean_{m,t} H(p[m,t,n])
U_intra[n]     = mean_m H(p_bar_basin[m,n])  -  U_aleatoric[n]
U_inter[n]     = H(p_bar[n])  -  mean_m H(p_bar_basin[m,n])
U_total[n]     = H(p_bar[n])
```

By construction `U_total = U_aleatoric + U_intra + U_inter` (checked with a runtime assertion
below). This is the standard total-entropy decomposition (Depeweg et al., 2018; Smith & Gal,
2018) generalized to a two-level (basin, pass) sampling scheme:

- `U_aleatoric` — irreducible data noise, present even with a single deterministic pass.
- `U_intra`     — width of the posterior *around a single basin* (what plain MC-Dropout measures).
- `U_inter`     — disagreement *between basins*, i.e. between distinct local optima (what a
  Deep Ensemble measures). This is the component MC-Dropout alone cannot see, since it only ever
  samples around one basin.

`U_epistemic = U_intra + U_inter`. For M2 (M=1, T=1) every term besides `U_aleatoric` is
identically zero. For M4 (T=1 per basin) `U_intra` is identically zero. The formula below handles
all four canonical models without special-casing.

## Outputs

`NB3-models/{dataset}/basin_{01..50}.pt`
`NB3-result/m{2,3,4,5}_uncertainty_{dataset}.csv` and `..._natural_{dataset}.csv`
`NB3-result/m{2,3,4,5}_summary_metrics_{dataset}.json` and `..._natural_{dataset}.json`
`NB3-result/training_log_{dataset}.csv`

Upload `NB3-models` as a Kaggle Dataset named `xai-credit-basins` — it is the input for NB4 and
NB5.


## 1. Installs & imports

In [ ]:
!pip install fastparquet scikit-learn -q

import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 2. Configuration

In [ ]:
class Config:
    INPUT_DIR = "/kaggle/input/xai-credit-preprocessed"
    OUTPUT_DIR = "/kaggle/working"
    SEED = 42
    DATASETS = ["home_credit", "taiwan", "gmsc"]

    N_BASINS = 50
    HIDDEN_DIMS = [256, 128, 64]
    DROPOUT_RATE = 0.2
    BATCH_SIZE = 1024
    LR = 1e-3
    WEIGHT_DECAY = 1e-5
    MAX_EPOCHS = 100
    PATIENCE = 15
    MIN_DELTA = 1e-4

    # canonical model inference budgets, all M x T = 50
    M3_BASINS, M3_PASSES = 1, 50
    M4_BASINS, M4_PASSES = 50, 1
    M5_BASINS, M5_PASSES = 5, 10

    # optional: point this at a Kaggle input dataset from a previous partial run of this
    # notebook to resume training across sessions (Kaggle working directories do not persist
    # between sessions). Leave as None for a fresh run.
    RESUME_INPUT_DIR = None

    if not os.path.exists(INPUT_DIR):
        print(f"WARNING: {INPUT_DIR} not found. Assuming local test run.")
        INPUT_DIR = "../kaggle_outputs/xai-credit-preprocessed"


for ds in Config.DATASETS:
    os.makedirs(f"{Config.OUTPUT_DIR}/models/{ds}", exist_ok=True)

import random
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)


## 3. Model definition

In [ ]:
class CreditMLP(nn.Module):
    """Standard tabular MLP: Linear -> BatchNorm -> ReLU -> Dropout, repeated, then a linear
    output head producing a single logit (sigmoid applied outside the model)."""

    def __init__(self, input_dim, hidden_dims=(256, 128, 64), dropout_rate=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)   # returns logits


def enable_mc_dropout(model):
    """Set the whole model to eval() (so BatchNorm uses running statistics, not batch
    statistics), then re-enable only the Dropout layers. This isolates MC-Dropout's stochasticity
    to dropout alone -- letting BatchNorm use batch statistics during MC sampling would inject an
    unrelated source of noise tied to which other samples happen to be in the batch."""
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()
    return model


## 4. Training loop for a single basin

Full per-epoch logging (train loss, val loss, val AUC, val AUPRC) so a diverging or clearly
under-performing basin is visible immediately and the notebook can be stopped early if needed.
Early stopping on validation AUC. Checkpoints are written as soon as training finishes for a
basin, and existing checkpoints are skipped on re-run (resumable across interrupted sessions).


In [ ]:
def train_one_basin(ds, basin_id, X_train, y_train, X_val, y_val, checkpoint_path):
    seed = Config.SEED + basin_id
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = CreditMLP(X_train.shape[1], Config.HIDDEN_DIMS, Config.DROPOUT_RATE).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
                               generator=torch.Generator().manual_seed(seed))

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)

    best_val_auc = -np.inf
    best_state = None
    epochs_no_improve = 0
    t0 = time.time()
    log_rows = []

    for epoch in range(1, Config.MAX_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        train_loss = epoch_loss / max(1, n_batches)

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits).cpu().numpy()
        val_loss = float(nn.functional.binary_cross_entropy(
            torch.sigmoid(val_logits).cpu(), torch.tensor(y_val, dtype=torch.float32)
        ))
        val_auc = roc_auc_score(y_val, val_probs)
        val_auprc = average_precision_score(y_val, val_probs)

        improved = val_auc > best_val_auc + Config.MIN_DELTA
        if improved:
            best_val_auc = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        log_rows.append({
            "basin_id": basin_id, "epoch": epoch, "train_loss": train_loss,
            "val_loss": val_loss, "val_auc": val_auc, "val_auprc": val_auprc,
            "best_val_auc_so_far": best_val_auc, "improved": improved,
        })

        if epoch == 1 or epoch % 5 == 0 or improved:
            print(f"    basin {basin_id:02d} | epoch {epoch:3d}/{Config.MAX_EPOCHS} "
                  f"| train_loss={train_loss:.5f} | val_loss={val_loss:.5f} "
                  f"| val_auc={val_auc:.5f} | val_auprc={val_auprc:.5f} "
                  f"| best={best_val_auc:.5f} | no_improve={epochs_no_improve}")

        if epochs_no_improve >= Config.PATIENCE:
            print(f"    basin {basin_id:02d} | EARLY STOP at epoch {epoch} "
                  f"(no improvement for {Config.PATIENCE} epochs). best_val_auc={best_val_auc:.5f}")
            break
    else:
        print(f"    basin {basin_id:02d} | reached MAX_EPOCHS={Config.MAX_EPOCHS} "
              f"without early stop. best_val_auc={best_val_auc:.5f}")

    elapsed = time.time() - t0
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), checkpoint_path)

    log_df = pd.DataFrame(log_rows)
    final_row = {
        "basin_id": basin_id, "final_epoch": log_df["epoch"].max(),
        "best_val_auc": best_val_auc,
        "best_val_auprc": log_df.loc[log_df["val_auc"].idxmax(), "val_auprc"],
        "early_stopped": bool(log_df["epoch"].max() < Config.MAX_EPOCHS),
        "train_time_sec": elapsed, "seed": seed,
    }
    print(f"    basin {basin_id:02d} | DONE in {elapsed:.1f}s | best_val_auc={best_val_auc:.5f}")
    return final_row


## 5. HB-MCD inference utilities

In [ ]:
def batched_mc_passes(model, X, T, dropout_on):
    """Returns array of shape (T, N): probability of class 1 for each of T forward passes."""
    if dropout_on:
        enable_mc_dropout(model)
    else:
        model.eval()
    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    out = np.zeros((T, len(X)))
    with torch.no_grad():
        for t in range(T):
            logits = model(X_t)
            out[t] = torch.sigmoid(logits).cpu().numpy()
    return out


def binary_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def hb_mcd_decompose(preds_mtn):
    """preds_mtn: array (M, T, N). Returns dict of (N,) arrays: pred_mean, aleatoric, intra,
    inter, epistemic, total. See notebook header for the formula and its derivation."""
    M, T, N = preds_mtn.shape
    p_bar_basin = preds_mtn.mean(axis=1)              # (M, N)
    p_bar = p_bar_basin.mean(axis=0)                  # (N,)

    aleatoric = binary_entropy(preds_mtn).mean(axis=(0, 1))       # (N,)
    intra = binary_entropy(p_bar_basin).mean(axis=0) - aleatoric  # (N,)
    inter = binary_entropy(p_bar) - binary_entropy(p_bar_basin).mean(axis=0)  # (N,)
    total = binary_entropy(p_bar)

    max_residual = np.max(np.abs(total - (aleatoric + intra + inter)))
    assert max_residual < 1e-6, f"HB-MCD decomposition does not sum to total! residual={max_residual}"

    return {
        "pred_mean": p_bar, "aleatoric_unc": aleatoric, "intra_unc": np.maximum(intra, 0.0),
        "inter_unc": np.maximum(inter, 0.0), "epistemic_unc": np.maximum(intra, 0.0) + np.maximum(inter, 0.0),
        "total_unc": total,
    }


def load_basin(ds, basin_id, input_dim):
    model = CreditMLP(input_dim, Config.HIDDEN_DIMS, Config.DROPOUT_RATE).to(DEVICE)
    path = f"{Config.OUTPUT_DIR}/models/{ds}/basin_{basin_id:02d}.pt"
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    return model


def build_preds_mtn(ds, basin_ids, T, dropout_on, X, input_dim):
    """Stacks MC passes across the given basin_ids into an (M, T, N) array."""
    M = len(basin_ids)
    out = np.zeros((M, T, len(X)))
    for i, bid in enumerate(basin_ids):
        model = load_basin(ds, bid, input_dim)
        out[i] = batched_mc_passes(model, X, T, dropout_on)
    return out


## 6. Compute predictive + calibration metrics for a decomposition

In [ ]:
def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob > bins[i]) & (y_prob <= bins[i + 1])
        if mask.sum() == 0:
            continue
        ece += mask.sum() / len(y_true) * abs(y_prob[mask].mean() - y_true[mask].mean())
    return float(ece)


def compute_nll(y_true, y_prob, eps=1e-10):
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return float(-np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob)))


def compute_brier(y_true, y_prob):
    return float(np.mean((y_prob - y_true) ** 2))


def save_model_outputs(model_name, ds, split_name, decomp, y_true):
    n = len(y_true)
    df = pd.DataFrame({
        "sample_id": np.arange(n), "y_true": y_true, "pred_mean": decomp["pred_mean"],
        "aleatoric_unc": decomp["aleatoric_unc"], "intra_unc": decomp["intra_unc"],
        "inter_unc": decomp["inter_unc"], "epistemic_unc": decomp["epistemic_unc"],
        "total_unc": decomp["total_unc"],
    })
    suffix = "" if split_name == "balanced" else "_natural"
    df.to_csv(f"{Config.OUTPUT_DIR}/{model_name}_uncertainty{suffix}_{ds}.csv", index=False)

    pred = decomp["pred_mean"]
    auroc = roc_auc_score(y_true, pred)
    auprc = average_precision_score(y_true, pred)
    ece = compute_ece(y_true, pred)
    nll = compute_nll(y_true, pred)
    brier = compute_brier(y_true, pred)
    epi_pct = float(np.mean(decomp["epistemic_unc"] / np.maximum(decomp["total_unc"], 1e-9)) * 100)

    metrics = {
        "predictive": {"auc": float(auroc), "auprc": float(auprc)},
        "calibration": {"ece": ece, "nll": nll, "brier": brier},
        "uncertainty_decomposition": {
            "mean_aleatoric": float(decomp["aleatoric_unc"].mean()),
            "mean_intra": float(decomp["intra_unc"].mean()),
            "mean_inter": float(decomp["inter_unc"].mean()),
            "mean_epistemic": float(decomp["epistemic_unc"].mean()),
            "mean_total": float(decomp["total_unc"].mean()),
            "epistemic_pct_of_total": epi_pct,
        },
        "n_samples": int(n), "prevalence": float(y_true.mean()),
    }
    with open(f"{Config.OUTPUT_DIR}/{model_name}_summary_metrics{suffix}_{ds}.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print(f"  [{model_name} / {split_name:>8}] AUC={auroc:.4f} ECE={ece:.4f} NLL={nll:.4f} "
          f"| epistemic={decomp['epistemic_unc'].mean():.5f} "
          f"({epi_pct:.2f}% of total) | intra={decomp['intra_unc'].mean():.5f} "
          f"| inter={decomp['inter_unc'].mean():.5f}")


## 7. Main per-dataset pipeline: train 50 basins, then derive M2-M5

In [ ]:
def process_dataset(ds):
    print(f"\n{'=' * 70}")
    print(f"DATASET: {ds}")
    print(f"{'=' * 70}")

    train_df = pd.read_parquet(f"{Config.INPUT_DIR}/{ds}_train.parquet")
    val_df = pd.read_parquet(f"{Config.INPUT_DIR}/{ds}_val.parquet")
    test_df = pd.read_parquet(f"{Config.INPUT_DIR}/{ds}_test_balanced.parquet")
    natural_df = pd.read_parquet(f"{Config.INPUT_DIR}/{ds}_test_natural.parquet")

    X_train = train_df.drop(columns=["TARGET"]).values.astype(np.float32)
    y_train = train_df["TARGET"].values.astype(np.float32)
    X_val = val_df.drop(columns=["TARGET"]).values.astype(np.float32)
    y_val = val_df["TARGET"].values.astype(np.float32)
    X_test = test_df.drop(columns=["TARGET"]).values.astype(np.float32)
    y_test = test_df["TARGET"].values.astype(np.float32)
    X_nat = natural_df.drop(columns=["TARGET"]).values.astype(np.float32)
    y_nat = natural_df["TARGET"].values.astype(np.float32)
    input_dim = X_train.shape[1]

    print(f"train={X_train.shape}  val={X_val.shape}  test_balanced={X_test.shape}  "
          f"test_natural={X_nat.shape}  input_dim={input_dim}")

    # --- 7.1 train N_BASINS independent basins ---
    print(f"\nTraining {Config.N_BASINS} basins...")
    log_rows = []
    for basin_id in range(1, Config.N_BASINS + 1):
        ckpt = f"{Config.OUTPUT_DIR}/models/{ds}/basin_{basin_id:02d}.pt"

        if Config.RESUME_INPUT_DIR is not None:
            resume_ckpt = f"{Config.RESUME_INPUT_DIR}/{ds}/basin_{basin_id:02d}.pt"
            if os.path.exists(resume_ckpt) and not os.path.exists(ckpt):
                import shutil
                shutil.copy(resume_ckpt, ckpt)

        if os.path.exists(ckpt):
            print(f"  basin {basin_id:02d} | checkpoint already exists, skipping training.")
            continue

        print(f"  basin {basin_id:02d} | training...")
        row = train_one_basin(ds, basin_id, X_train, y_train, X_val, y_val, ckpt)
        log_rows.append(row)

    if log_rows:
        log_df = pd.DataFrame(log_rows)
        log_path = f"{Config.OUTPUT_DIR}/training_log_{ds}.csv"
        if os.path.exists(log_path):
            log_df = pd.concat([pd.read_csv(log_path), log_df], ignore_index=True)
        log_df.to_csv(log_path, index=False)
        print(f"\nTraining log saved -> {log_path}")
        print(log_df[["basin_id", "final_epoch", "best_val_auc", "early_stopped",
                       "train_time_sec"]].describe().to_string())

    # --- 7.2 derive canonical M2 / M3 / M4 / M5 ---
    print(f"\nDeriving canonical models for {ds}...")
    for split_name, Xs, ys in [("balanced", X_test, y_test), ("natural", X_nat, y_nat)]:
        print(f"\n  -- split: {split_name} (n={len(ys)}) --")

        preds_m2 = build_preds_mtn(ds, [1], T=1, dropout_on=False, X=Xs, input_dim=input_dim)
        decomp_m2 = hb_mcd_decompose(preds_m2)
        save_model_outputs("m2", ds, split_name, decomp_m2, ys)

        preds_m3 = build_preds_mtn(ds, [1], T=Config.M3_PASSES, dropout_on=True, X=Xs, input_dim=input_dim)
        decomp_m3 = hb_mcd_decompose(preds_m3)
        save_model_outputs("m3", ds, split_name, decomp_m3, ys)

        preds_m4 = build_preds_mtn(ds, list(range(1, Config.M4_BASINS + 1)), T=Config.M4_PASSES,
                                    dropout_on=False, X=Xs, input_dim=input_dim)
        decomp_m4 = hb_mcd_decompose(preds_m4)
        save_model_outputs("m4", ds, split_name, decomp_m4, ys)

        preds_m5 = build_preds_mtn(ds, list(range(1, Config.M5_BASINS + 1)), T=Config.M5_PASSES,
                                    dropout_on=True, X=Xs, input_dim=input_dim)
        decomp_m5 = hb_mcd_decompose(preds_m5)
        save_model_outputs("m5", ds, split_name, decomp_m5, ys)

    print(f"\nFinished {ds}.")


## 8. Run for all datasets

In [ ]:
for ds in Config.DATASETS:
    try:
        process_dataset(ds)
    except Exception as e:
        import traceback
        print(f"FAILED on {ds}: {e}")
        traceback.print_exc()

print("\nAll datasets processed.")
print("Upload the 'models' folder as a Kaggle Dataset named 'xai-credit-basins' "
      "(rename to NB3-models/{dataset}/basin_XX.pt locally).")
print("Everything else in /kaggle/working goes to NB3-result.")
